# Z3 Primer: choices, constraints, and better answers

**CS 271P · Optional PA 4 supplement · About 20 minutes · Nothing to submit**

Z3 can choose values that satisfy rules you write. Its Python interface lets
you express those rules without writing the search procedure yourself. You
still decide what the variables mean and which rules the problem requires.

We will choose exhibits for a small science booth. This simpler problem teaches
the commands you can reuse in Weighted Stable Garden. The last section helps
you plan that transfer; you will write the garden's constraint model yourself.

**Start here:**

1. Use Python 3.10 or newer. From this notebook's folder, install the supplied
   dependency with `python3 -m pip install -r requirements.txt`.
2. Open this notebook in Jupyter, JupyterLab, or VS Code and select that Python
   environment. Choose **Run All**, then read the short explanations and outputs.
3. Try the two small changes in section 6. No additional packages or widgets
   are needed for the pictures.

If importing Z3 fails in your notebook kernel, run `%pip install -r requirements.txt`
in a new code cell in this folder, then restart the kernel and Run All. The
package is named **`z3-solver`**; the Python import is **`z3`**.


In [ ]:
import z3

print("Z3 version:", z3.get_version_string())
# This notebook uses the same pinned Z3 dependency as PA 4.


## 1. Give each decision a Boolean variable

An exhibit is either chosen or left out. A **Boolean variable** represents that
choice. `z3.Bool("booth_robot")` creates a symbolic true/false variable; it has
no selected value yet. Z3 chooses its value when we ask for a solution.

Two calls using the same name refer to the same Z3 variable. Give different
choices different names. The Python dictionary below is just a convenient way
to store those variables.

Our booth has these default options:

| Exhibit | Space used if chosen | Value if chosen |
|---|---:|---:|
| Robot | 2 | 5 |
| Poster | 1 | 3 |
| Puzzle | 2 | 4 |
| Screen | 3 | 8 |

There are **4 units of space**. We may choose **at most two exhibits**. We must
choose a poster or a puzzle (or both). A screen needs a poster explaining it.
The robot and screen cannot both use the booth's single electrical outlet.

**Predict:** which legal combination seems most valuable?


In [ ]:
# These are ordinary Python data. Section 6 changes just these two settings.
CAPACITY = 4
SCREEN_VALUE = 8

names = ("robot", "poster", "puzzle", "screen")
space = {"robot": 2, "poster": 1, "puzzle": 2, "screen": 3}
values = {"robot": 5, "poster": 3, "puzzle": 4, "screen": SCREEN_VALUE}

# These are Z3 decision variables, one for each exhibit.
picked = {name: z3.Bool(f"booth_{name}") for name in names}
print("Robot decision:", picked["robot"])  # A symbol, not a chosen value.


In [ ]:
# GIVEN: display helper. Read the Z3 cells; no need to edit this helper.
from html import escape

FORCE_TEXT = False  # Optional fallback if your frontend does not show HTML.
try:
    from IPython.display import HTML, display
except ImportError:
    display = None


def show_choices(choices=None, title="Available exhibits"):
    '''Draw four labeled cards; also work in a plain Python interpreter.'''
    if FORCE_TEXT or display is None:
        print(title)
        for name in names:
            state = "not decided" if choices is None else ("chosen" if choices[name] else "left out")
            print(f"  {name}: {state}; space {space[name]}; value {values[name]}")
        if choices is not None:
            print("Space used:", sum(space[n] * choices[n] for n in names), "/", CAPACITY)
        return

    cards = []
    for name in names:
        chosen = choices is not None and choices[name] == 1
        state = "not decided" if choices is None else ("chosen" if chosen else "left out")
        background, foreground = ("#276b77", "white") if chosen else ("#edf4f6", "#20343d")
        cards.append(
            f'<div style="flex:1 1 130px;max-width:210px;border:1px solid #9cb6bc;'
            f'border-radius:6px;padding:12px;background:{background};color:{foreground}">'
            f'<strong>{escape(name.title())}</strong><br>{state}<br>'
            f'Space: {space[name]} · Value: {values[name]}</div>'
        )
    totals = ""
    if choices is not None:
        used = sum(space[n] * choices[n] for n in names)
        worth = sum(values[n] * choices[n] for n in names)
        totals = f'<p>Space used: <strong>{used}/{CAPACITY}</strong> · Total value: <strong>{worth}</strong></p>'
    display(HTML(f'<div><strong>{escape(title)}</strong>'
                 f'<div style="display:flex;flex-wrap:wrap;gap:8px;margin-top:8px">'
                 + "".join(cards) + '</div>' + totals + '</div>'))


show_choices()


## 2. Express rules and totals

Use Z3's logical commands to combine symbolic choices:

| Command | Meaning in this example |
|---|---|
| `z3.Or(a, b)` | At least one of `a`, `b` is true; both may be true. |
| `z3.And(a, b)` | Both are true. |
| `z3.Not(a)` | `a` is false. |
| `z3.Implies(a, b)` | If `a` is true, `b` must be true. It does not require `a` to be true. |

**Use these commands to build constraints.** Python's `and`, `or`, `not`, and
`if` do not build symbolic Z3 rules. Ordinary Python loops and comprehensions
are useful for *constructing* lists of rules and expressions.

To add up choices, turn each true/false choice into an integer expression:

- `z3.If(choice, 1, 0)` contributes one if chosen, zero otherwise.
- `z3.If(choice, weight, 0)` contributes its weight if chosen, zero otherwise.
- `z3.Sum([...])` adds those expressions.

`z3.If` constructs a conditional expression. It does not choose a branch yet.
The familiar comparisons `<=`, `>`, `==`, and `!=` also construct constraints
when applied to Z3 expressions. `=` is Python assignment; `==` expresses equality.


In [ ]:
# Count chosen exhibits and express their total space and value.
chosen_count = z3.Sum([z3.If(picked[n], 1, 0) for n in names])
used_space = z3.Sum([z3.If(picked[n], space[n], 0) for n in names])
total_value = z3.Sum([z3.If(picked[n], values[n], 0) for n in names])

# A Python list of mandatory rules. These define which selections are legal.
rules = [
    chosen_count <= 2,
    used_space <= CAPACITY,
    z3.Or(picked["poster"], picked["puzzle"]),
    z3.Implies(picked["screen"], picked["poster"]),
    z3.Not(z3.And(picked["robot"], picked["screen"])),
]

print("Space expression:", used_space)
# At this point it is still symbolic. A model will give it a concrete value.


## 3. Ask for one legal selection

`z3.Solver()` creates a solver. `solver.add(...)` gives it mandatory constraints;
it accepts a single rule, several rules, or a list such as `rules` above.
`solver.check()` then attempts to satisfy all the rules together.

| Result | What it tells you | Next step |
|---|---|---|
| `z3.sat` | Z3 found a selection satisfying this model. | Read it with `solver.model()`. |
| `z3.unsat` | No selection satisfies all the current constraints. | Revisit the rules or the question you asked. |
| `z3.unknown` | Z3 did not decide the question, for example because of a timeout. | Keep any earlier checked answer; do not claim impossibility or optimality. |

Set a call timeout with `solver.set(timeout=1000)`: the units are **milliseconds**.
This limits a Z3 call; it does not measure the whole time spent constructing and
running your Python program. PA 4's overall timer is explained in the handout.

**A plain `Solver` finds a feasible selection; it does not automatically
maximize `total_value`.** Merely defining that expression does not request optimization.


In [ ]:
booth = z3.Solver()
booth.set(timeout=1000)  # Allow up to one second for a Z3 call.
booth.add(rules)

status = booth.check()
print("Status:", status)
# Only call booth.model() in the SAT branch below.


## 4. Read the model as ordinary Python data

A **model** is one assignment of values to the Z3 variables that satisfies
the constraints. `model.eval(expression)` reads a variable or expression in
that assignment.

`model.eval(choice, model_completion=True)` supplies a concrete Boolean even
if Z3 left that choice unspecified. This fills in unspecified values; it
does **not** add missing problem rules. `z3.is_true(...)` converts a Z3 Boolean
value to a Python Boolean, and `int(...)` makes the ordinary integer 0 or 1.
For an integer result, `.as_long()` converts it to an ordinary Python integer.

Look at the picture and check the rules yourself. A model is a solution to the
constraints you supplied; if you forgot a rule, Z3 cannot enforce it for you.


In [ ]:
if status == z3.sat:
    model = booth.model()
    choices = {
        n: int(z3.is_true(model.eval(picked[n], model_completion=True)))
        for n in names
    }
    value = model.eval(total_value).as_long()
    print("Plain Python 0/1 choices:", choices)
    print("Value of this feasible selection:", value)
    show_choices(choices, "One legal selection — not necessarily the best")
elif status == z3.unsat:
    print("No selection satisfies these booth rules.")
else:
    print("Search was inconclusive:", booth.reason_unknown())


## 5. Improve the value by asking for a better answer

Once you have a legal selection of value `best_value`, add
`total_value > best_value` and ask again. Each SAT result is a strictly better
selection. Save its **decoded Python data** before changing the solver again.

The loop below starts with a fresh solver and the same original `rules`, so
rerunning this cell does not retain an old improvement constraint.

- If the next result is **SAT**, save that better selection and continue.
- If it is **UNSAT after a saved selection**, no strictly better selection
  satisfies this model. That proves the saved selection optimal for the model.
- If it is **unknown**, stop and keep the saved selection. You have no proof
  that it is optimal.

The booth has only four choices, so this finite improvement loop is small.
For PA 4, also check the overall remaining budget: several individually timed
calls can together exceed three seconds. Leave time to decode, check, and save
an answer. See page 5 of the Submission Instructions for `budget` and saved offers.


In [ ]:
# Fresh solver: rebuild from the original booth rules, not a previous search.
search = z3.Solver()
search.set(timeout=1000)
search.add(rules)

best_choices = None
best_value = None
improvement_values = []
proved_optimal = False

while True:
    result = search.check()
    if result == z3.sat:
        model = search.model()
        # Copy this answer into Python data before the next search.
        best_choices = {
            n: int(z3.is_true(model.eval(picked[n], model_completion=True)))
            for n in names
        }
        best_value = model.eval(total_value).as_long()
        improvement_values.append(best_value)
        search.add(total_value > best_value)  # Next answer must be better.
    elif result == z3.unsat:
        # With an earlier answer, UNSAT rules out a strictly better answer.
        proved_optimal = best_choices is not None
        break
    else:
        print("Stopped without a proof:", search.reason_unknown())
        break  # Keep the saved Python data, if any.

print("Values found:", improvement_values)  # The intermediate sequence may vary.
if best_choices is not None:
    print("Best value:", best_value, "| Proved optimal:", proved_optimal)
    show_choices(best_choices, "Best saved booth selection")
else:
    print("No feasible selection was found; final status:", result)


## 6. Try two small changes

All the supplied examples already work. These two edits are enough to practice;
you do not need to implement a new function or submit an answer.

1. **Predict:** with only 3 units of space, which exhibits would you choose?
   Set `CAPACITY = 3` in section 1 and Run All. Compare the best-selection picture
   and value with your prediction.
2. Reset `CAPACITY = 4`, then set `SCREEN_VALUE = 2` and Run All. Which exhibits
   form the best selection now? Why does feasibility alone not
   tell you which selection is best?

Changing the data cell requires rerunning later cells: they rebuild expressions
and constraints from those data. **Run All** handles that for you.

<details><summary>Check your predictions after trying the edits</summary>

With the default settings, poster + screen has value **11** and uses 4 units.
With capacity 3 and the default values, robot + poster has value **8**.
With capacity 4 and screen value 2, robot + puzzle has value **9**.
The sequence of intermediate SAT answers can vary; compare the final value.

</details>


## Optional: test a temporary restriction with `push` and `pop`

`push()` marks a point in the solver's constraint stack. After you add a
temporary rule, `pop()` removes rules added since that matching `push`.

Here we temporarily demand both the robot and the screen, contradicting the
single-outlet rule. Removing that demand restores the original booth rules.
With the default settings, the statuses are **sat → unsat → sat**. The UNSAT
result applies to the temporary question, not to the original problem.

This is useful for experiments. You do not need it for the improvement loop above.


In [ ]:
trial = z3.Solver()
trial.set(timeout=1000)
trial.add(rules)
print("Original rules:", trial.check())

trial.push()  # Remember the original constraints.
trial.add(picked["robot"], picked["screen"])  # Require both for this trial.
print("Temporary demand:", trial.check())
trial.pop()  # Remove the temporary demand; keep all the original rules.

print("Restored rules:", trial.check())


## 7. Use the same commands in PA 4

You now have the building blocks for a first direct Z3 model. Return to the
garden rules on page 2 of the Submission Instructions and the `solve` interface
on page 3. Use this checklist while writing your own model:

| What you learned here | What to work out for the garden |
|---|---|
| One `Bool` per exhibit | Decide how to represent each tile's live/dead choice, with unique variable names. |
| `If` and `Sum` build counts | Construct the neighbor counts the garden rules need. |
| `And`, `Or`, `Not`, `Implies`, and comparisons express hard rules | Translate every stability rule, including rules for dead positions, blocked tiles, and the outside ring. |
| A weighted sum expresses value | Express the total weight of live tiles. |
| `model.eval`, `is_true`, and `int` produce 0/1 choices | Return the required list of rows of ordinary Python integer 0/1 values. |
| A SAT result gives a candidate | Check it with the supplied `problem.check_output(instance, board)` before relying on it. |
| Ask for a better value and retain saved answers | Respect the full `budget`; save checked improvements with `budget.offer` and return your best legal board. |

Begin with a tiny public case, and check legality before focusing on value.
The garden's empty board is a legal fallback; arrange to offer it before
attempting a longer search. The booth's constraints are specific to exhibits:
use the garden handout to decide which constraints your submitted program needs.

The next step is **your own garden model**, followed by a small test:

```sh
python3 test_local.py --case 4 --compare
```

Edit and submit **`solver.py` only**. This primer and its pictures are not submitted.


## Quick reference

| Command | Purpose |
|---|---|
| `z3.Bool("unique_name")` | Create a symbolic true/false choice. |
| `z3.And(...)`, `z3.Or(...)`, `z3.Not(...)`, `z3.Implies(a, b)` | Combine symbolic conditions. |
| `z3.If(condition, yes_value, no_value)` | Build a conditional expression, such as a choice's contribution. |
| `z3.Sum(list_of_expressions)` | Build a total or count. |
| `solver = z3.Solver()` | Create a satisfiability solver. |
| `solver.add(rule)` or `solver.add(list_of_rules)` | Add mandatory constraints. |
| `solver.set(timeout=milliseconds)` | Set the timeout for Z3 calls. |
| `status = solver.check()` | Try to satisfy the current constraints. |
| `model = solver.model()` | Get the satisfying assignment after SAT. |
| `model.eval(expr, model_completion=True)` | Evaluate an expression, completing unspecified values if needed. |
| `int(z3.is_true(boolean_value))`, `integer_value.as_long()` | Convert Z3 values into ordinary Python integers. |
| `solver.push()`, `solver.pop()` | Add and remove a scope of temporary constraints. |

**Other functionality you can explore later:** `z3.Int` for integer decisions,
`z3.Optimize` with `maximize` for optimization objectives, pseudo-Boolean
constraints such as `z3.PbLe`, `solver.unsat_core()` with tracked constraints
for diagnosis, and `solver.statistics()` for inspecting search effort. Those
features and advanced garden strategies are outside this primer.

The examples use the PA's pinned Z3 package. For additional details, see the
[Z3 Python API reference](https://z3prover.github.io/api/html/namespacez3py.html),
especially [Solver](https://z3prover.github.io/api/html/classz3py_1_1_solver.html)
and [ModelRef](https://z3prover.github.io/api/html/classz3py_1_1_model_ref.html).
